# Example 12.6.3 / 3: Recourse feasibility

Companion notebook for **12.6.3 Two-Stage Recourse and Nonanticipativity** in the AMPL lab Documentation, section "Recourse feasibility". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Recourse feasibility

Relatively complete recourse means every allowed first-stage decision has feasible recourse for every modeled scenario. If this is not true, the model needs first-stage restrictions, explicit emergency actions with justified penalties, or an algorithm capable of generating feasibility cuts.

The order model shows the difference. This version must serve all demand, from the order or from
emergency units that cost 12 each and earn nothing, and the supplier delivers at most 120 units.
The run first fixes `Emergency` at 0 with `ampl.eval` (amplpy's `fix()` acts on one variable, not on
an indexed family), then frees it again with `unfix`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS;                          # demand outcomes
param probability {SCENARIOS} >= 0;     # scenario weight
param demand {SCENARIOS} >= 0;          # units
param unit_cost >= 0;                   # dollars per unit ordered
param price >= 0;                       # dollars per unit sold from the order
param max_order >= 0;                   # units the supplier can deliver
param emergency_cost >= 0;              # dollars per emergency unit

var Order >= 0 <= max_order;            # first stage
var Sell {SCENARIOS} >= 0;              # second stage: units sold from the order
var Emergency {SCENARIOS} >= 0;         # second stage: emergency units

maximize ExpectedProfit:
    - unit_cost * Order
  + sum {s in SCENARIOS} probability[s]
      * (price * Sell[s] - emergency_cost * Emergency[s]);

subject to SellFromOrder {s in SCENARIOS}: Sell[s] <= Order;
# every unit of demand is served
subject to ServeDemand {s in SCENARIOS}: Sell[s] + Emergency[s] = demand[s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SCENARIOS"] = ["low", "mid", "high"]
ampl.param["probability"] = {"low": 0.3, "mid": 0.4, "high": 0.3}
ampl.param["demand"] = {"low": 40, "mid": 100, "high": 130}
ampl.param["unit_cost"] = 4
ampl.param["price"] = 10
ampl.param["max_order"] = 120
ampl.param["emergency_cost"] = 12

In [5]:
# run.py
ampl.eval("fix {s in SCENARIOS} Emergency[s] := 0;")  # no emergency units
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)

ampl.eval("unfix Emergency;")
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["Order"].value(), ampl.obj["ExpectedProfit"].value())
print(ampl.var["Emergency"].to_dict())

infeasible
solved 120.0 364.0
{'high': 10, 'low': 0, 'mid': 0}


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set SCENARIOS := low mid high;
param probability := low 0.3  mid 0.4  high 0.3;
param demand := low 40  mid 100  high 130;
param unit_cost := 4;
param price := 10;
param max_order := 120;
param emergency_cost := 12;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
# This step fails on purpose, as on the page; the error is printed so the notebook goes on
try:
    ampl.eval(r"""
# model.run
option solver highs;
fix {s in SCENARIOS} Emergency[s] := 0;
solve;
display solve_result;
unfix Emergency;
solve;
display Order, ExpectedProfit, Emergency;
""")
except Exception:
    pass  # AMPL has printed the error above

[WARNING] 
	presolve, variable Order:
		impossible deduced bounds: lower = 130, upper = 120;
		difference = 10


solve_result = infeasible

HiGHS 1.14.0: optimal solution; objective 364
0 simplex iterations
0 barrier iterations
Order = 120
ExpectedProfit = 364

Emergency [*] :=
high  10
 low   0
 mid   0
;

